# Sprint 3 — final test riêng
Chỉ mở sau khi chọn model trên dev và khóa đủ resources/code/checkpoint. Dùng runtime đã chuẩn bị trong notebook train/dev; không train ở đây. Gold ZIP chỉ giải nén tại cell scorer sau khi predictions đã đóng băng.


In [ ]:
from pathlib import Path
import json, subprocess, os, sys
ROOT = Path("/content/DACN")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from src.modeling.colab_handoff import safe_extract
from src.evaluation.dev_runner import file_hash
PY = str(ROOT / ".venv_colab311/bin/python")
ENABLE_FINAL_TEST = False
HANDOFF = Path("/content/final_evaluation_manifest.json")
CONFIG = Path("data/interim/modeling/sprint03/CHOSEN_selection/final_model_config.json")
SELECTION = CONFIG.with_name("selection_lock.json")
RUN = Path("data/processed/evaluation/sprint03/CHOSEN_final_test_run001")
BACKUP_DIRECTORY = None  # persistent directory configured before final execution


In [ ]:
# Input-only bundle; no gold is opened in preflight or inference.
if ENABLE_FINAL_TEST:
    if BACKUP_DIRECTORY is None:
        raise RuntimeError("Configure a persistent backup before final execution")
    handoff = json.loads(HANDOFF.read_text())
    if handoff["status"] != "FINAL_EVALUATION_BUNDLES_READY":
        raise RuntimeError("Test gold release pending")
    INPUT_ZIP = Path("/content") / Path(handoff["input_archive"]["path"]).name
    safe_extract(INPUT_ZIP, ROOT, handoff["input_archive"]["sha256"], "test_input_bundle_manifest.json")
    CORPUS = Path(handoff["corpus_path"])
    COMMON = ["--input", str(CORPUS / "test_input.jsonl"), "--corpus-manifest", str(CORPUS / "manifest.json"),
              "--model-config", str(CONFIG), "--selection-lock", str(SELECTION)]
    subprocess.run([PY, "-m", "scripts.55_run_span_test", "preflight"] + COMMON, check=True)
    subprocess.run([PY, "-m", "scripts.55_run_span_test", "infer", "--output-dir", str(RUN),
                    "--execute-final-test"] + COMMON, check=True)


In [ ]:
# Predictions and their manifest already exist; only now is scorer gold extracted.
if ENABLE_FINAL_TEST:
    if not (RUN / "run_manifest.json").is_file():
        raise RuntimeError("Freeze prediction before opening gold")
    GOLD_ZIP = Path("/content") / Path(handoff["gold_archive"]["path"]).name
    safe_extract(GOLD_ZIP, ROOT, handoff["gold_archive"]["sha256"], "test_gold_bundle_manifest.json")
    subprocess.run([PY, "-m", "scripts.56_score_span_test", "--run-dir", str(RUN),
        "--gold", str(CORPUS / "test_benchmark_t0.jsonl"), "--input", str(CORPUS / "test_input.jsonl"),
        "--corpus-manifest", str(CORPUS / "manifest.json"), "--execute-final-test"], check=True)
    from src.modeling.colab_runtime import backup_run
    backup_run(RUN, BACKUP_DIRECTORY)


Mỗi model/run mới dùng lock đã chốt trên dev. Ablation dùng config/lock cùng checkpoint của proposed. Không đổi mapping/threshold sau xem test. 5 trường báo track riêng; fixture và smoke không được đưa vào bảng kết quả pretrained.
